# Electivo de Bioinformática — Clase 8

## RNA-Seq I: recuento de transcritos/genes y normalización (CPM, TPM y *size factors*)

**Programa:** Doctorado — 2º año
**Duración:** 2,5 horas

### Objetivos de la clase

Al finalizar esta clase, serán capaces de:

1. Explicar qué es un **recuento** en RNA-seq: qué lecturas se cuentan para un gen, cuáles no, y por qué.
2. Generar una tabla de recuentos a partir de BAMs alineados con **`featureCounts`**, y leer su resumen (asignadas, ambiguas, sin gen, multimapeo).
3. Explicar por qué los recuentos crudos **no se pueden comparar directamente** (profundidad de secuenciación, largo del gen, composición de la librería).
4. Calcular e interpretar **CPM**, **RPKM/FPKM** y **TPM**, y saber para qué sirve cada una.
5. Calcular a mano los **size factors** de DESeq2 (mediana de razones) y entender por qué la expresión diferencial parte de los recuentos crudos y no de TPM.
6. Explorar las muestras con una **PCA** y dejar listo el objeto de DESeq2 para la clase de expresión diferencial.

---
## 1. Dónde estamos y qué vamos a hacer hoy (5 min)

Hasta ahora trabajamos con **ADN**: alineamos lecturas a una referencia y buscamos en qué posiciones la muestra **difiere** de ella (variantes). En RNA-seq la pregunta cambia: ya no nos interesa *qué base* hay en cada posición, sino **cuántas lecturas vienen de cada gen**, porque eso refleja cuánto ARN de ese gen había en la muestra.

El flujo típico de un RNA-seq es:

```
FASTQ ──► QC ──► alineamiento ──► BAM ──► RECUENTO ──► matriz de recuentos ──► normalización ──► expresión diferencial
          (FastQC)  (STAR, HISAT2)          (featureCounts)   (genes × muestras)    (CPM, TPM, DESeq2)    (próxima clase)
```

El QC y el alineamiento ya los conocen (clases 4 a 6). La única diferencia importante es que en RNA-seq el alineador debe ser **consciente del splicing** (*splice-aware*): una lectura puede caer mitad en un exón y mitad en el siguiente, saltándose el intrón. Por eso **hoy partimos directamente de BAMs ya alineados** y nos concentramos en lo nuevo: **contar y normalizar**.

**Los datos de hoy:** el estudio *airway* (Himes et al., *PLoS One* 2014; GEO GSE52778). Cuatro líneas celulares de músculo liso de vía aérea humana, cada una **sin tratar** (`untrt`) y **tratada con dexametasona** (`trt`), un glucocorticoide usado en asma: **8 muestras**, RNA-seq pareado. Es uno de los conjuntos de datos más usados para enseñar expresión diferencial y viene empaquetado en Bioconductor, así que no hay que descargar nada a mano.

---
## 2. Preparar el ambiente `rnaseq` (10 min, y sigue instalándose mientras vemos la teoría)

Hoy necesitamos herramientas nuevas, así que creamos un ambiente nuevo:

| Paquete | Para qué |
|---|---|
| `subread` | Trae `featureCounts`, el programa que cuenta lecturas por gen |
| `samtools` | Para mirar los BAM (ya lo conocen) |
| `r-base` | El lenguaje R |
| `bioconductor-deseq2` | El paquete de R que usaremos para normalizar hoy y para la expresión diferencial la próxima clase |
| `bioconductor-airway` | Los datos del estudio *airway*: la matriz de recuentos completa y 8 BAMs pequeños de ejemplo |

Primero, una verificación rápida de que `conda` sigue disponible (lo instalamos en la clase 3):

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda --version
conda env list

Ahora creamos el ambiente. **Esto puede tardar entre 5 y 15 minutos** (R y Bioconductor son pesados): déjenlo corriendo y sigamos con la sección 3 mientras tanto.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda create -y -n rnaseq -c conda-forge -c bioconda \
    subread samtools r-base bioconductor-deseq2 bioconductor-airway

> Si la instalación de `bioconductor-airway` falla (es un paquete de datos que se descarga desde Bioconductor al instalarse), creen el ambiente sin él y usen el **Apéndice** al final del notebook para bajar los mismos archivos directamente.

---
## 3. Fundamentos del recuento (20 min)

### 3.1 ¿Qué es un "recuento"?

Un **recuento** es simplemente el **número de lecturas (o fragmentos, si es pareado) que se alinearon sobre los exones de un gen**. Para cada muestra obtenemos un número por gen; juntando todas las muestras queda una **matriz de recuentos**: filas = genes, columnas = muestras, valores = enteros.

```
                 muestra1  muestra2  muestra3  ...
ENSG00000096060       254      5191       323
ENSG00000120129       650      5602      1028
...
```

Para decidir a qué gen pertenece cada lectura, el programa de recuento necesita dos cosas: el **BAM** (dónde cayó cada lectura) y una **anotación** en formato **GTF** (dónde están los exones de cada gen; lo vimos en la clase 3).

### 3.2 Qué se cuenta y qué no

Para cada lectura (o par de lecturas), un programa como `featureCounts` decide:

| Situación de la lectura | ¿Se cuenta? | Cómo aparece en el resumen de `featureCounts` |
|---|---|---|
| Cae dentro de los exones de **un solo** gen | Sí, **1** para ese gen | `Assigned` |
| Cruza de un exón a otro del mismo gen (lectura con *splicing*) | Sí, **1** (no 2) para ese gen | `Assigned` |
| Cae en una región **sin genes** anotados (intergénica) | No | `Unassigned_NoFeatures` |
| Solapa exones de **dos genes** distintos | No, por defecto (no sabemos de cuál viene) | `Unassigned_Ambiguity` |
| Se alineó igual de bien en **varios lugares** del genoma | No, por defecto | `Unassigned_MultiMapping` |

Dos detalles más que importan:

- **Pareado (paired-end):** las dos lecturas de un par vienen del **mismo fragmento** de ARN, así que se cuentan como **un** fragmento, no como dos.
- **Hebra (strandedness):** muchos protocolos de librería conservan la información de qué hebra se transcribió. Si el protocolo es *stranded*, una lectura en la hebra contraria a la del gen **no** debe contarse para ese gen. Hay que decírselo al programa (`-s 0` no stranded, `-s 1` stranded, `-s 2` stranded inverso) — si se elige mal, se pierde la mitad o más de las lecturas. Hoy veremos cómo verificarlo con los datos.

### 3.3 Nivel de gen vs. nivel de transcrito

Un gen puede tener varias **isoformas** (transcritos) que comparten exones. Si una lectura cae en un exón compartido, no hay forma directa de saber de qué isoforma vino. Por eso hay dos estrategias:

- **Recuento por gen** (`featureCounts`, `HTSeq`): junta todos los exones del gen y cuenta lecturas sobre esa unión. Simple, robusto, suficiente para la mayoría de los análisis de expresión diferencial. **Es lo que haremos hoy.**
- **Cuantificación por transcrito** (`salmon`, `kallisto`): usan un modelo estadístico para **repartir** las lecturas ambiguas entre isoformas. Entregan *recuentos estimados* (con decimales) y TPM por transcrito. Son muy rápidos porque ni siquiera hacen un alineamiento completo (*pseudoalineamiento*). Se pueden sumar después a nivel de gen.

### 3.4 La idea clave de toda la clase

El número de lecturas que obtiene un gen **no depende solo de cuánto se expresa**. Aproximadamente:

$$\text{recuento} \;\propto\; \text{expresión (n.º de moléculas de ARN)} \;\times\; \text{largo del gen} \;\times\; \text{profundidad de secuenciación}$$

- **Profundidad:** si secuenciamos el doble de lecturas en una muestra, *todos* sus genes tendrán aproximadamente el doble de recuentos, sin que nada biológico haya cambiado.
- **Largo:** un ARN de 10 kb se rompe en más fragmentos que uno de 1 kb, así que con la **misma** cantidad de moléculas produce ~10 veces más lecturas.

**Normalizar** es corregir estos factores técnicos para que los números reflejen biología. Eso es lo que haremos en las secciones 6 y 7.

---
## 4. Del BAM a la tabla de recuentos con `featureCounts` (25 min)

El paquete `airway` incluye 8 BAMs **pequeños** (uno por muestra), con solo las lecturas que caen en una región de 400 kb del cromosoma 1, y un GTF con los 20 genes de esa región. Son ideales para ver cómo se genera una tabla de recuentos en segundos.

### 4.1 Copiar los datos de ejemplo a nuestra carpeta de trabajo

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate rnaseq

mkdir -p ~/bioinfo/clase8/datos ~/bioinfo/clase8/resultados ~/bioinfo/clase8/graficos
cd ~/bioinfo/clase8

# Carpeta donde quedaron los archivos de ejemplo del paquete airway
EXTDATA=$(Rscript -e 'cat(system.file("extdata", package = "airway"))')
echo "Archivos de ejemplo en: $EXTDATA"

cp "$EXTDATA"/*_subset.bam "$EXTDATA"/Homo_sapiens.GRCh37.75_subset.gtf "$EXTDATA"/sample_table.csv datos/
ls -lh datos/

La tabla de muestras nos dice qué es cada archivo (`cell` = línea celular, `dex` = tratamiento):

In [ ]:
%%bash
cd ~/bioinfo/clase8
cut -d, -f1,3,4 datos/sample_table.csv

### 4.2 Un vistazo rápido al BAM y al GTF

Estos BAM fueron alineados con **STAR**, un alineador *splice-aware*. Busquemos lecturas con una `N` en el CIGAR: la `N` significa "salto en la referencia" — la lectura cruzó un intrón.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate rnaseq

cd ~/bioinfo/clase8
echo "Lecturas totales:        $(samtools view -c datos/SRR1039508_subset.bam)"
echo "Lecturas con splicing:   $(samtools view datos/SRR1039508_subset.bam | awk '$6 ~ /N/' | wc -l)"
echo
samtools view datos/SRR1039508_subset.bam | awk '$6 ~ /N/' | head -3 | cut -f1-6

Por ejemplo, `57M972N6M` = 57 bases alineadas, un salto de **972 pb** (un intrón), y 6 bases más alineadas en el exón siguiente. Más de una cuarta parte de las lecturas cruzan un intrón: un alineador de ADN como `bwa` las alinearía mal.

En el GTF, cada gen tiene sus transcritos y cada transcrito sus exones. Por ejemplo, el gen `TARDBP`:

In [ ]:
%%bash
cd ~/bioinfo/clase8
GTF=datos/Homo_sapiens.GRCh37.75_subset.gtf
echo "Genes en el GTF: $(awk -F'\t' '$3=="gene"' $GTF | wc -l)"
echo "TARDBP -> transcritos: $(grep 'gene_name "TARDBP"' $GTF | awk -F'\t' '$3=="transcript"' | wc -l), exones: $(grep 'gene_name "TARDBP"' $GTF | awk -F'\t' '$3=="exon"' | wc -l)"

`TARDBP` tiene 10 transcritos y 50 exones (muchos repetidos entre isoformas). Para contar a nivel de gen, `featureCounts` agrupa por `gene_id` todos los exones y usa su **unión**.

### 4.3 ¿Es una librería *stranded*? Probemos las tres opciones

No siempre sabemos cómo se preparó la librería. Una forma práctica de averiguarlo: contar una muestra con `-s 0`, `-s 1` y `-s 2`, y comparar cuántas lecturas se asignan.

Opciones que usamos:
- `-a`: el GTF de anotación
- `-o`: archivo de salida
- `-p --countReadPairs`: datos pareados; contar **fragmentos** (pares), no lecturas sueltas
- `-s`: hebra (0, 1 o 2)

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate rnaseq

cd ~/bioinfo/clase8
for S in 0 1 2; do
    echo -n "-s $S  ->  "
    featureCounts -p --countReadPairs -s $S \
        -a datos/Homo_sapiens.GRCh37.75_subset.gtf \
        -o resultados/prueba_s$S.txt \
        datos/SRR1039508_subset.bam 2>&1 | grep "Successfully assigned"
done

Resultado esperado: ~**93%** con `-s 0` y ~**50%** con `-s 1` y con `-s 2`.

¿Cómo se lee? Si la librería fuera *stranded*, una de las opciones (`-s 1` o `-s 2`) asignaría casi tanto como `-s 0`, y la otra casi nada. Acá ambas dan ~50%: las lecturas caen **mitad y mitad** en cada hebra, o sea, la librería **no es stranded**. Usamos `-s 0`.

### 4.4 Contar las 8 muestras

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate rnaseq

cd ~/bioinfo/clase8
featureCounts -p --countReadPairs -s 0 \
    -a datos/Homo_sapiens.GRCh37.75_subset.gtf \
    -o resultados/recuentos_subset.txt \
    datos/*_subset.bam

`featureCounts` genera dos archivos. Primero el **resumen** (`.summary`): qué pasó con cada fragmento. Mostramos solo las categorías que no están en cero:

In [ ]:
%%bash
cd ~/bioinfo/clase8
sed 's/datos\///g; s/_subset.bam//g' resultados/recuentos_subset.txt.summary \
    | awk 'NR==1 { print; next } { s = 0; for (i = 2; i <= NF; i++) s += $i; if (s > 0) print }'

Las filas corresponden a la tabla de la sección 3.2: la gran mayoría queda `Assigned`, y el resto se descarta por caer fuera de genes (`NoFeatures`), en dos genes a la vez (`Ambiguity`) o en varios lugares del genoma (`MultiMapping`). **Revisar este resumen es el control de calidad del recuento:** si `Assigned` fuera, digamos, 40%, habría que sospechar del parámetro `-s`, de la anotación (¿mismo ensamblaje del genoma que el usado para alinear?) o de contaminación con ADN genómico.

Ahora la **tabla de recuentos**. La primera línea es un comentario con el comando usado; luego vienen las columnas `Geneid Chr Start End Strand Length` y una columna por muestra. Mostramos `Geneid`, `Length` y los recuentos:

In [ ]:
%%bash
cd ~/bioinfo/clase8
grep -v "^#" resultados/recuentos_subset.txt | cut -f1,6- | sed 's/datos\///g; s/_subset.bam//g'

Tres cosas para notar:

1. **Muchos genes tienen 0 o casi 0** (pseudogenes, ARN pequeños). Es normal: en una matriz completa, cerca de la mitad de los genes no se expresan en un tejido dado.
2. La columna **`Length`** es el largo de la **unión de exones** del gen (no del gen completo con intrones). La vamos a necesitar para calcular TPM.
3. Miren `ENSG00000171819` (gen **`ANGPTL7`**): en cada línea celular tiene muy pocos recuentos sin tratar y muchos más con dexametasona (4→50, 19→543, 14→1067). ¿Es esto ya evidencia de expresión diferencial? Todavía no podemos decirlo: primero hay que ver si las muestras tenían la misma profundidad. Eso es la normalización.

---
## 5. La matriz de recuentos completa en R (15 min)

Los BAM de ejemplo cubren solo 20 genes. El paquete `airway` también trae la **matriz completa** (63.677 genes × 8 muestras), generada por los autores con el mismo principio que acabamos de ver sobre los BAM completos. Desde aquí trabajamos en **R**.

**Cómo trabajar con las celdas de R:**
1. En la terminal del servidor: `conda activate rnaseq` y luego escriban `R` (Enter). El prompt cambia a `>`.
2. Copien el contenido de cada celda `%%R` **sin la línea `%%R`** y péguenlo en esa sesión de R.
3. A diferencia de las celdas `%%bash`, **la sesión de R conserva todo lo que hayan creado** mientras no la cierren. Si se cierra, vuelvan a correr las celdas desde esta sección.
4. Para salir de R al final: `q()` y respondan `n`.

Cargamos los paquetes y los datos:

In [ ]:
%%R
suppressPackageStartupMessages(library(DESeq2))
library(ggplot2)
library(airway)
data(airway)

# Los resultados de hoy se guardan en la carpeta de la clase
setwd("~/bioinfo/clase8")

# Que "untrt" (sin tratar) sea el grupo de referencia
airway$dex <- relevel(airway$dex, "untrt")

airway

`airway` es un objeto `SummarizedExperiment`: una "caja" que guarda juntas la matriz de recuentos (`assay`), la información de las muestras (`colData`) y la de los genes (`rowData`, `rowRanges`). Saquemos cada parte:

In [ ]:
%%R
cts <- assay(airway)          # matriz de recuentos: genes x muestras
dim(cts)
cts[1:5, ]

colData(airway)[, c("cell", "dex")]   # diseño del experimento

Antes de normalizar, tres preguntas sobre los recuentos crudos:

In [ ]:
%%R
# 1) ¿Cuántos fragmentos se asignaron en cada muestra? (en millones)
round(colSums(cts) / 1e6, 1)

# 2) ¿Cuántos genes no tienen ni una lectura en ninguna muestra?
sum(rowSums(cts) == 0)

# 3) ¿Qué tan desigual es la distribución?
summary(cts[, 1])
top <- sort(rowSums(cts), decreasing = TRUE)
round(sum(top[1:100]) / sum(top), 2)   # fracción de lecturas que se llevan los 100 genes más expresados

Lo que deberían ver:

1. Las muestras van de **~15 a ~31 millones** de fragmentos: la muestra `SRR1039517` tiene el **doble** de profundidad que `SRR1039513`. Comparar recuentos crudos entre ellas sería injusto.
2. ~**30.000** genes (casi la mitad) no tienen lecturas.
3. La mediana del recuento es **0**, la media ~324 y el máximo ~300.000: una distribución extremadamente asimétrica. Los **100 genes más expresados se llevan ~30% de todas las lecturas**. Esto será importante en la sección 7.

---
## 6. Normalización I: profundidad y largo — CPM, RPKM y TPM (25 min)

### 6.1 Las fórmulas

| Unidad | Corrige | Cálculo |
|---|---|---|
| **CPM** (*counts per million*) | Profundidad | $\text{CPM} = \dfrac{\text{recuento}}{\text{total de la muestra}} \times 10^6$ |
| **RPKM / FPKM** (*reads/fragments per kilobase per million*) | Profundidad, luego largo | $\text{RPKM} = \dfrac{\text{recuento}}{\text{largo (kb)} \times \text{total (millones)}}$ |
| **TPM** (*transcripts per million*) | Largo, luego profundidad | $\text{RPK} = \dfrac{\text{recuento}}{\text{largo (kb)}}$ ;  $\text{TPM} = \dfrac{\text{RPK}}{\sum \text{RPK de la muestra}} \times 10^6$ |

RPKM (lecturas sueltas) y FPKM (fragmentos de pares) son lo mismo con distinto nombre. TPM usa exactamente los mismos ingredientes que RPKM, pero en **orden inverso**: primero corrige por largo y después por profundidad. Ese cambio de orden tiene una consecuencia importante: **en cada muestra, los TPM suman siempre 1.000.000**. Por eso un TPM se lee como: *"de cada millón de transcritos presentes en esta muestra, cuántos son de este gen"*.

### 6.2 Un ejemplo de juguete

Tres genes con **la misma cantidad de moléculas** de ARN, pero distinto largo: A (2 kb), B (4 kb), C (1 kb). La muestra S2 se secuenció el doble de profundo que S1.

In [ ]:
%%R
toy <- matrix(c(10, 20, 5,     # S1
                20, 40, 10),   # S2 (el doble de profundidad)
              ncol = 2, dimnames = list(c("A", "B", "C"), c("S1", "S2")))
largo_toy <- c(A = 2000, B = 4000, C = 1000)
toy

# CPM: corrige la profundidad
cpm_toy <- t(t(toy) / colSums(toy)) * 1e6
round(cpm_toy)

# TPM: primero largo, después profundidad
rpk_toy <- toy / (largo_toy / 1000)
tpm_toy <- t(t(rpk_toy) / colSums(rpk_toy)) * 1e6
round(tpm_toy)

- Con **CPM**, S1 y S2 ya son iguales (se corrigió la profundidad), pero B parece el doble de expresado que A, y A el doble que C. Es un efecto **del largo**, no de la biología.
- Con **TPM**, los tres genes quedan en 333.333: la conclusión correcta, porque los tres tenían la misma cantidad de moléculas.

> Nota sobre `t(t(x) / colSums(x))`: en R, dividir una matriz por un vector divide **por filas**; trasponer, dividir y volver a trasponer es el truco para dividir **cada columna** por su propio total.

### 6.3 Lo mismo en los datos reales

Para TPM necesitamos el **largo de cada gen**, igual que la columna `Length` de `featureCounts`: la unión de sus exones. Lo sacamos de `rowRanges(airway)`, que tiene las coordenadas de los exones de cada gen:

In [ ]:
%%R
largo <- sum(width(reduce(rowRanges(airway))))   # unión de exones por gen, en pb
summary(largo)

# Comprobación: es el mismo "Length" que nos dio featureCounts para TARDBP
largo["ENSG00000120948"]

(Debería ser **4622**, igual que en la tabla de la sección 4.4.)

Ahora calculamos las tres unidades para toda la matriz:

In [ ]:
%%R
cpm  <- t(t(cts) / colSums(cts)) * 1e6
rpkm <- cpm / (largo / 1000)
rpk  <- cts / (largo / 1000)
tpm  <- t(t(rpk) / colSums(rpk)) * 1e6

# ¿Cuánto suma cada columna?
round(colSums(rpkm))
colSums(tpm)

Las sumas de **TPM** son todas exactamente 1.000.000; las de **RPKM** cambian de muestra en muestra (~258.000 a ~288.000). Eso significa que **un RPKM de 10 no representa la misma proporción del transcriptoma en todas las muestras** — por eso hoy se prefiere TPM a RPKM/FPKM.

Miremos algunos genes conocidos. Los primeros siete se sabe que responden a glucocorticoides (`CRISPLD2` es el hallazgo principal del artículo original); `ACTB` y `GAPDH` son genes "de mantención" (*housekeeping*) que se usan a menudo como referencia:

In [ ]:
%%R
genes <- c(FKBP5 = "ENSG00000096060", TSC22D3 = "ENSG00000157514", PER1 = "ENSG00000179094",
           DUSP1 = "ENSG00000120129", KLF15 = "ENSG00000163884", ZBTB16 = "ENSG00000109906",
           CRISPLD2 = "ENSG00000103196", ACTB = "ENSG00000075624", GAPDH = "ENSG00000111640")

tabla_tpm <- round(tpm[genes, ], 1)
rownames(tabla_tpm) <- names(genes)
tabla_tpm

Las columnas alternan **sin tratar / tratada** para cada línea celular (08/09, 12/13, 16/17, 20/21). Ya se ve el patrón: `FKBP5`, `ZBTB16`, `KLF15`… suben muchísimo con dexametasona en las cuatro líneas.

**Para qué sirve TPM:** comparar **genes distintos dentro de una misma muestra** ("en esta muestra, ¿qué se expresa más, `GAPDH` o `DUSP1`?") y reportar niveles de expresión en tablas o figuras. Es lo que entregan directamente `salmon` y `kallisto`, y lo que verán en bases de datos como GTEx.

> Un detalle técnico: `salmon`/`kallisto` usan un **largo efectivo** (el largo del transcrito menos el tamaño medio del fragmento), que es más correcto que el largo bruto que usamos aquí. La idea es la misma.

---
## 7. Normalización II: el problema de composición y los *size factors* de DESeq2 (20 min)

CPM y TPM tienen un supuesto escondido: que **el total de la muestra** es una buena medida de su profundidad. Eso falla cuando unos pocos genes cambian mucho.

### 7.1 Ejemplo de juguete: un solo gen cambia

Cuatro genes. Entre S1 y S2, **solo D cambia** (sube 7 veces); A, B y C están exactamente igual.

In [ ]:
%%R
comp <- matrix(c(100, 100, 100, 100,    # S1
                 100, 100, 100, 700),   # S2: solo D sube
               ncol = 2, dimnames = list(c("A", "B", "C", "D"), c("S1", "S2")))
comp

# CPM
round(t(t(comp) / colSums(comp)) * 1e6)

Con CPM, **A, B y C parecen haber bajado 2,5 veces** en S2, aunque no cambiaron en absoluto: como D se "llevó" más lecturas del total, a los demás les tocó una fracción menor. Esto se llama **sesgo de composición**, y con datos reales (donde los 100 genes más expresados se llevan el 30% de las lecturas, sección 5) es un problema real. Si usáramos CPM o TPM para buscar genes diferenciales, encontraríamos muchos falsos positivos.

### 7.2 La solución de DESeq2: mediana de razones

La idea de DESeq2 es: *"la mayoría de los genes no cambia entre condiciones"*. Entonces, en lugar de usar el total, estima la profundidad de cada muestra con la **mediana** de muchas comparaciones gen por gen, que es robusta a unos pocos genes que cambian mucho.

Pasos:
1. Para cada gen, calcular una **muestra de referencia "promedio"**: la media geométrica de sus recuentos en todas las muestras.
2. Para cada muestra, dividir el recuento de cada gen por ese valor de referencia (una **razón** por gen).
3. El ***size factor*** de la muestra es la **mediana** de esas razones.
4. Recuentos normalizados = recuentos / *size factor*.

In [ ]:
%%R
log_geo <- rowMeans(log(comp))                         # paso 1 (en log, la media geométrica es una media simple)
razones <- exp(log(comp) - log_geo)                    # paso 2
razones
sf_toy <- apply(razones, 2, median)                    # paso 3
sf_toy
t(t(comp) / sf_toy)                                    # paso 4

Las razones de A, B y C son 1 en ambas muestras; solo la de D se sale. La mediana ignora a D, ambos *size factors* quedan en 1, y los recuentos normalizados muestran lo correcto: **solo D cambió**.

### 7.3 *Size factors* en los datos reales: a mano y con DESeq2

Con datos reales hay un detalle: si un gen tiene 0 en alguna muestra, su media geométrica es 0 (en log, `-Inf`), así que esos genes se dejan fuera del cálculo.

In [ ]:
%%R
# A mano
log_geo <- rowMeans(log(cts))
usar    <- is.finite(log_geo)            # solo genes sin ceros
sum(usar)
sf_manual <- apply(cts[usar, ], 2, function(x) exp(median(log(x) - log_geo[usar])))

# Con DESeq2
dds <- DESeqDataSet(airway, design = ~ cell + dex)
dds <- estimateSizeFactors(dds)

# Comparación (y, como referencia, el tamaño de librería relativo)
lib <- colSums(cts)
round(rbind(a_mano    = sf_manual,
            DESeq2    = sizeFactors(dds),
            total_rel = lib / exp(mean(log(lib)))), 3)

Las dos primeras filas son **idénticas**: eso es exactamente lo que hace `estimateSizeFactors`. La tercera fila (el total de cada muestra, escalado para comparar) se parece, pero no es igual: por ejemplo, para `SRR1039508` el total diría 0,96 y la mediana de razones dice 1,02. La diferencia es justamente el efecto de composición.

Dos cosas importantes sobre el objeto `dds` que acabamos de crear:

- **`design = ~ cell + dex`** le dice a DESeq2 que el experimento tiene dos factores: la línea celular y el tratamiento. Lo usaremos en la próxima clase.
- **DESeq2 guarda los recuentos crudos** y usa los *size factors* **dentro** de su modelo estadístico. Por eso la entrada de un análisis de expresión diferencial son siempre **recuentos crudos (enteros)**, nunca TPM, CPM ni recuentos ya normalizados.

Los recuentos normalizados, para mirar o graficar:

In [ ]:
%%R
nc <- counts(dds, normalized = TRUE)
tabla_nc <- round(nc[genes, ])
rownames(tabla_nc) <- names(genes)
tabla_nc

### 7.4 Resumen: ¿qué unidad uso para qué?

| Pregunta | Unidad adecuada |
|---|---|
| ¿Qué gen se expresa más **dentro de una muestra**? | **TPM** |
| ¿Cómo cambia **un mismo gen entre muestras**? (para mirar o graficar) | **Recuentos normalizados** de DESeq2 (o CPM, con cuidado por la composición) |
| ¿Qué genes cambian **significativamente** entre condiciones? (prueba estadística) | **Recuentos crudos** + *size factors* dentro del modelo (DESeq2, edgeR) |
| Comparar con otros estudios o bases de datos (GTEx) | **TPM** (con cautela: distintos protocolos y anotaciones) |

---
## 8. Mirar las muestras: transformación y PCA (10 min)

Antes de buscar genes diferenciales conviene mirar las muestras en conjunto: ¿se agrupan por tratamiento? ¿hay alguna muestra rara? Para eso usamos una **PCA** (análisis de componentes principales), que resume miles de genes en unos pocos ejes.

La PCA no funciona bien con recuentos directos: los genes muy expresados (que además tienen mucha más varianza) dominarían todo. Por eso primero se **transforma** a una escala tipo logaritmo. DESeq2 trae la función `vst()` (*variance stabilizing transformation*), que hace esto de forma más estable que un simple `log2(x + 1)` en los genes con pocos recuentos.

In [ ]:
%%R
vsd <- vst(dds, blind = TRUE)

# Coordenadas de cada muestra en los dos primeros componentes
pca <- plotPCA(vsd, intgroup = c("dex", "cell"), returnData = TRUE)
pca[, c("PC1", "PC2", "dex", "cell")]
round(attr(pca, "percentVar") * 100)    # % de la variación explicada por PC1 y PC2

# Gráficos en un PDF
pdf("graficos/clase8_exploracion.pdf", width = 8, height = 5)
par(mfrow = c(1, 2), mar = c(8, 4, 3, 1))
expr <- rowSums(cts) >= 10            # solo genes con algo de expresion (si no, la mitad son ceros)
boxplot(log2(cts[expr, ] + 1), las = 2, main = "log2(recuentos crudos + 1)")
boxplot(log2(nc[expr, ] + 1),  las = 2, main = "log2(normalizados + 1)")
pv <- round(attr(pca, "percentVar") * 100)
print(ggplot(pca, aes(PC1, PC2, color = dex, shape = cell)) +
        geom_point(size = 4) +
        labs(x = paste0("PC1 (", pv[1], "%)"), y = paste0("PC2 (", pv[2], "%)"),
             title = "PCA de las 8 muestras (vst)") +
        theme_bw())
dev.off()

Para ver el PDF, descárguenlo a su computador con `scp` desde una terminal **local** (igual que en la clase 3, con su configuración de conexión al clúster):

```bash
scp <usuario>@<servidor>:~/bioinfo/clase8/graficos/clase8_exploracion.pdf .
```

Qué deberían ver:

- **PC1 (~41% de la variación)** separa perfectamente **tratadas** (PC1 positivo) de **sin tratar** (PC1 negativo): la dexametasona es el efecto más grande del experimento.
- **PC2 (~26%)** separa las **líneas celulares** (por ejemplo, `N080611` arriba y `N061011` abajo), con la muestra tratada y la sin tratar de cada línea a la misma altura.
- Los boxplots de recuentos normalizados quedan más alineados entre muestras que los crudos.

Que la línea celular explique tanta variación es la razón por la que el diseño es `~ cell + dex` y no solo `~ dex`: así, en la próxima clase, DESeq2 compara cada muestra tratada **con su propia línea celular sin tratar** (un diseño pareado), en vez de mezclar la variación entre líneas con el efecto del tratamiento.

---
## 9. Ejercicio integrador (en parejas, 15 min)

Respondan en palabras (no hace falta entregar código). Las tablas que necesitan son `tabla_tpm` (sección 6.3) y `tabla_nc` (sección 7.3) — si hace falta, vuelvan a imprimirlas en R.

1. **Dentro de una muestra.** En `SRR1039508`, ¿qué gen se expresa más, `GAPDH` o `DUSP1`? ¿Cuántas veces más? ¿Qué unidad usaron y por qué esa y no recuentos crudos?

2. **Entre muestras.** Calculen, para cada línea celular, la razón tratada / sin tratar de `FKBP5` usando recuentos normalizados. En R:
   ```r
   tratadas    <- c("SRR1039509", "SRR1039513", "SRR1039517", "SRR1039521")
   sin_tratar  <- c("SRR1039508", "SRR1039512", "SRR1039516", "SRR1039520")
   round(log2(tabla_nc[, tratadas] / tabla_nc[, sin_tratar]), 2)
   ```
   (`log2` de la razón: 1 = el doble, 2 = cuatro veces, −1 = la mitad.) ¿El aumento de `FKBP5` es consistente en las cuatro líneas?

3. **¿Son estables los genes *housekeeping*?** Miren `ACTB` y `GAPDH` en la misma tabla. ¿Se mantienen constantes con el tratamiento? ¿Qué pasaría si normalizáramos todo el experimento usando solo `ACTB` como referencia (como se hace a veces en qPCR)? ¿Por qué DESeq2 usa la mediana de **miles** de genes en vez de uno?

4. **Pensamiento crítico.** Una compañera propone hacer la expresión diferencial de la próxima clase comparando con una prueba t los **TPM** de tratadas vs. sin tratar. Den dos razones, basadas en lo que vimos hoy, por las que esto no es lo recomendado.

5. **Diseño.** Según la PCA, ¿qué factores explican PC1 y PC2? Si en lugar de 4 líneas celulares hubiéramos tenido 8 donantes distintos, cada uno con solo una muestra (4 tratados, 4 sin tratar, todos de personas distintas), ¿qué sería más difícil de detectar y por qué?

### Guardar el trabajo para la próxima clase

Antes de salir de R, guarden el objeto de DESeq2 y la tabla de TPM:

In [ ]:
%%R
saveRDS(dds, "resultados/dds_airway.rds")

tpm_tabla <- data.frame(gene_id = rownames(tpm), gene_name = rowData(airway)$gene_name,
                        round(tpm, 3), check.names = FALSE)
write.table(tpm_tabla, "resultados/airway_tpm.tsv", sep = "\t", quote = FALSE, row.names = FALSE)

list.files("resultados")

Y salimos de R con `q()` (respondan `n` a la pregunta de guardar el *workspace*: ya guardamos lo que necesitamos).

---
## 10. Cierre y resumen (5 min)

### Comandos vistos hoy

```
Ambiente:        conda create -n rnaseq -c conda-forge -c bioconda subread samtools r-base bioconductor-deseq2 bioconductor-airway
Splicing:        samtools view archivo.bam | awk '$6 ~ /N/'                (lecturas que cruzan un intrón)
Hebra:           featureCounts -p --countReadPairs -s 0|1|2 -a anot.gtf -o salida.txt muestra.bam
Recuento:        featureCounts -p --countReadPairs -s 0 -a anot.gtf -o recuentos.txt *.bam
                 (revisar siempre recuentos.txt.summary)
```

```r
# R / DESeq2
cts   <- assay(airway)                                 # matriz de recuentos
largo <- sum(width(reduce(rowRanges(airway))))         # largo = unión de exones
cpm   <- t(t(cts) / colSums(cts)) * 1e6
rpk   <- cts / (largo / 1000);  tpm <- t(t(rpk) / colSums(rpk)) * 1e6
dds   <- DESeqDataSet(airway, design = ~ cell + dex)
dds   <- estimateSizeFactors(dds);  sizeFactors(dds)
nc    <- counts(dds, normalized = TRUE)
vsd   <- vst(dds, blind = TRUE);    plotPCA(vsd, intgroup = c("dex", "cell"))
```

### Ideas clave

- Un **recuento** = fragmentos asignados a los exones de un gen. Se descartan los ambiguos, los multimapeados y los que no caen en genes; el `.summary` de `featureCounts` es su control de calidad.
- Recuento ∝ expresión × **largo** × **profundidad** (× **composición**).
- **CPM** corrige profundidad; **TPM** corrige largo y profundidad y siempre suma 10⁶ por muestra → para comparar genes **dentro** de una muestra.
- Los ***size factors*** de DESeq2 (mediana de razones) corrigen profundidad **y** composición → para comparar **entre** muestras.
- La expresión diferencial parte siempre de **recuentos crudos**.

### Lo que queda listo para la próxima clase

- Ambiente `rnaseq` con R y DESeq2.
- `~/bioinfo/clase8/resultados/dds_airway.rds`: el objeto de DESeq2 con recuentos, diseño `~ cell + dex` y *size factors*. En la próxima clase lo cargamos con `readRDS()` y corremos directamente `DESeq()`.
- `~/bioinfo/clase8/resultados/airway_tpm.tsv`: tabla de TPM con nombre de gen.

### Tarea

1. Terminen las preguntas del ejercicio integrador.
2. Con `airway_tpm.tsv` y lo visto en bash: ¿cuáles son los **10 genes con mayor TPM** en la muestra `SRR1039508`? (Pista: `sort -t$'\t' -k3,3gr`, y recuerden saltarse el encabezado.) ¿Qué tipo de genes son? ¿Reconocen algunos de las clases anteriores? (Pista: miren los nombres que empiezan con `MT-`.) Algunos son pseudogenes (`...P5`, `...P6`): con lo que vimos en la sección 3.2 sobre lecturas multimapeadas, ¿por qué podría un pseudogeno aparecer tan expresado?

---
## Apéndice: si `bioconductor-airway` no se pudo instalar

Creen el ambiente sin ese paquete y bajen los mismos archivos desde el espejo del repositorio de Bioconductor en GitHub:

```bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda create -y -n rnaseq -c conda-forge -c bioconda subread samtools r-base bioconductor-deseq2

mkdir -p ~/bioinfo/clase8/datos && cd ~/bioinfo/clase8/datos
BASE=https://raw.githubusercontent.com/bioconductor-source/airway/devel
wget -q $BASE/data/airway.RData
wget -q $BASE/inst/extdata/sample_table.csv
wget -q $BASE/inst/extdata/Homo_sapiens.GRCh37.75_subset.gtf
for M in 08 09 12 13 16 17 20 21; do
    wget -q $BASE/inst/extdata/SRR10395${M}_subset.bam
done
ls -lh
```

Con esto, en la sección 4.1 **omitan la celda de copia** (los archivos ya están en `datos/`) y en la sección 5 reemplacen `library(airway)` y `data(airway)` por:

```r
load("~/bioinfo/clase8/datos/airway.RData")
```